
# Week 9 — Day 1
# Sprint 4 Planning, Serialization & MLOps

## Melanoma Skin Cancer — Benign vs Malignant

**Project:** Melanoma Skin Lesion Classification  
**Sprint:** Sprint 4 — Deployment  
**Day:** Day 1 — Sprint 4 Planning, Serialization & MLOps

### Purpose

This notebook starts Sprint 4 by taking the validated Week 8 CNN pipeline and preparing it for deployment.

The Week 8 Day 5 project used:

- Image size: **128 × 128 × 3**
- Classes: **Benign = 0, Malignant = 1**
- Preprocessing: **OpenCV → BGR to RGB → resize → float32 → divide by 255**
- Model: **CNN with Conv2D(32), Conv2D(64), Dense(128), Dropout(0.3), Sigmoid**
- Reference threshold: **0.50**
- Candidate deployment threshold: **0.35**, selected by the highest F1 score in Day 5




##  Learning Objectives

By the end of Day 1, we will:

1. Complete Sprint 4 planning and define the deployment backlog.
2. Carry forward the Sprint 3 retrospective improvement.
3. Document the exact Week 8 model and preprocessing contract.
4. Set reproducibility seeds.
5. Save the trained CNN using Keras serialization.
6. Save deployment metadata, including the selected threshold.
7. Preserve any fitted preprocessing objects if they exist.
8. Reload the saved model and reproduce a known prediction.
9. Freeze the deployment environment in `requirements.txt`.
10. Log the deployment artifact/configuration with MLflow.
11. Create a deployment manifest ready for the next Sprint 4 days.



#  Sprint 4 Planning

## Sprint 4 Goal

**Deploy the trained melanoma classification model as a live public application.**

The Week 8 Day 5 Sprint 4 preview describes the transition as:

```text
Sprint 3
Preprocessing → CNN → Evaluation → Error Analysis → Explainability

                 ↓

Sprint 4
Saved Model → Prediction API / App → User Input
→ Same Preprocessing → Prediction → Documentation
```

The most important engineering requirement is to keep the serving pipeline identical to the validated training pipeline.



## Deployment Backlog

| Priority | Task | Day 1 |
|---|---|---|
| High | Define Sprint 4 deployment goal |x|
| High | Serialize trained CNN |x|
| High | Preserve preprocessing contract |x|
| High | Save model metadata |x|
| High | Verify save → load → predict |x|
| High | Freeze deployment requirements |x|
| Medium | MLflow tracking |x|
| High | Build prediction API / serving layer | Next |
| High | Build user interface | Next |
| High | Integrate model and UI | Next |
| High | End-to-end testing | Next |
| High | Deploy to public URL | Next |
| Medium | Final documentation and polish | Next |



## Sprint 3 Retrospective → Sprint 4 Improvement

The Week 8 Day 5 retrospective identified the following concrete Sprint 4 change:

> Package the preprocessing and trained model into a usable prediction interface, preserve the exact preprocessing used during training, and document limitations clearly.

Therefore, Day 1 will explicitly save the **preprocessing contract** and the **decision threshold** alongside the model.

This prevents a common deployment problem: **training/serving skew**.



#  Why Deployment Matters

A model that works only inside a notebook is not yet a usable product.

Deployment turns the trained model into something that can accept a real user's image and return a prediction.

The intended Sprint 4 flow is:


User uploads image
        ↓
OpenCV / RGB / Resize 128×128 / float32 / 255
        ↓
Serialized CNN
        ↓
Malignant probability
        ↓
Threshold = 0.35
        ↓
Benign / Malignant

The final goal of Sprint 4 is a **live public application** built around this verified model.



#  Week 8 Final Model Contract

The Day 5 notebook confirms the exact pipeline used during Sprint 3:

### Input

- Image
- Resize to **128 × 128**
- 3 RGB channels

### Preprocessing


OpenCV read
    ->
BGR → RGB
    ->
Resize to 128 × 128
    ->
float32
    ->
pixel / 255.0


### Class mapping


Benign = 0
Malignant = 1


### Model

```text
Conv2D(32, 3×3, ReLU)
→ MaxPooling2D
→ Conv2D(64, 3×3, ReLU)
→ MaxPooling2D
→ Flatten
→ Dense(128, ReLU)
→ Dropout(0.3)
→ Dense(1, Sigmoid)
```

### Decision rule

Week 8 used `0.50` as the reference threshold and selected `0.35` as the candidate operating threshold because it gave the highest F1 among the tested thresholds.

At `0.35`:

- Precision (Malignant): **0.8746**
- Recall (Malignant): **0.9140**
- F1 (Malignant): **0.8939**
- False Positives: **131**
- False Negatives: **86**
- ROC-AUC: **0.9517**
- PR-AUC: **0.9445**


#  Imports and Reproducibility


In [1]:


import os
import json
import random
import platform
from pathlib import Path

import numpy as np
import tensorflow as tf

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Python:", platform.python_version())
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("Seed:", SEED)


Python: 3.13.15
TensorFlow: 2.20.0
NumPy: 2.1.3
Seed: 42


# Exact Week 8 Deployment Configuration


In [2]:


IMG_SIZE = (128, 128)
CHANNELS = 3

CLASS_NAMES = ["Benign", "Malignant"]
CLASS_TO_LABEL = {"Benign": 0, "Malignant": 1}

DEFAULT_THRESHOLD = 0.50
SELECTED_THRESHOLD = 0.35

NORMALIZATION = "pixel / 255.0"
COLOR_FORMAT = "RGB"
INPUT_LIBRARY = "OpenCV"

ARTIFACT_DIR = Path("deployment_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Image size:", IMG_SIZE)
print("Class mapping:", CLASS_TO_LABEL)
print("Reference threshold:", DEFAULT_THRESHOLD)
print("Selected deployment threshold:", SELECTED_THRESHOLD)


Image size: (128, 128)
Class mapping: {'Benign': 0, 'Malignant': 1}
Reference threshold: 0.5
Selected deployment threshold: 0.35



# Day 5 Evaluation Results to Carry into Deployment

These values are not newly calculated here. They are the validated results recorded in the Week 8 Day 5 notebook.

At the reference threshold `0.50`, the model achieved:

- Accuracy: **0.8755**
- Precision (Malignant): **0.9243**
- Recall (Malignant): **0.8180**
- F1 (Malignant): **0.8679**
- ROC-AUC: **0.9517**
- PR-AUC: **0.9445**
- False Positives: **67**
- False Negatives: **182**

At the selected threshold `0.35`, the model achieved:

- Accuracy: **0.8915**
- Precision (Malignant): **0.8746**
- Recall (Malignant): **0.9140**
- F1 (Malignant): **0.8939**
- ROC-AUC: **0.9517**
- PR-AUC: **0.9445**
- False Positives: **131**
- False Negatives: **86**

The threshold changes the **decision rule**, not the trained CNN weights.

Because false negatives are more critical for this task, the selected threshold gives substantially higher malignant recall and fewer missed malignant cases.



# Locate the Trained Week 8 Model




In [9]:
import os

print(os.path.exists("/content/melanoma_cnn.keras"))
print(os.listdir("/content"))

True
['.config', 'melanoma_cnn.keras', 'deployment_artifacts', 'sample_data']


In [10]:
from tensorflow.keras.models import load_model
SOURCE_MODEL_PATH = "/content/melanoma_cnn.keras"
model = load_model(SOURCE_MODEL_PATH)
print("Week 8 trained model loaded successfully.")
print(model)

Week 8 trained model loaded successfully.
<Sequential name=sequential, built=True>


In [11]:

from tensorflow.keras.models import load_model
if model is None and SOURCE_MODEL_PATH:
    source_path = Path(SOURCE_MODEL_PATH)
    if not source_path.exists():
        raise FileNotFoundError(f"Model not found: {source_path}")
    model = load_model(source_path)
    print("Loaded model from:", source_path)
if model is not None:
    print("Model input shape:", model.input_shape)
    print("Model output shape:", model.output_shape)
else:
    print("Model is not loaded yet.")


Model input shape: (None, 128, 128, 3)
Model output shape: (None, 1)



#  Verify the Model Architecture

The Week 8 Day 5 model used:

- Conv2D(32)
- MaxPooling2D
- Conv2D(64)
- MaxPooling2D
- Flatten
- Dense(128)
- Dropout(0.3)
- Dense(1, sigmoid)

The model was approximately **7.39 million trainable parameters**.

The check below confirms the loaded model has the expected input/output dimensions.


In [12]:

if model is not None:
    model.summary()

    expected_input = (None, 128, 128, 3)

    print("\nExpected input:", expected_input)
    print("Actual input:  ", model.input_shape)

    if tuple(model.input_shape) != expected_input:
        raise ValueError(
            "Model input shape does not match the Week 8 Day 5 contract."
        )

    if tuple(model.output_shape) != (None, 1):
        raise ValueError(
            "Model output shape does not match the expected binary classifier."
        )

    print("\nPASS: model input/output shapes match the Week 8 contract.")
else:
    print("Load the trained model before running this verification.")


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 57600)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     7,372,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,177,349 (84.60 MB)

 Trainable params: 7,392,449 (28.20 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 14,784,900 (56.40 MB)


Expected input: (None, 128, 128, 3)
Actual input:   (None, 128, 128, 3)

PASS: model input/output shapes match the Week 8 contract.



# Serialize the Trained CNN

For TensorFlow/Keras, the model will be saved in the `.keras` format.

This produces the production model artifact:

```text
deployment_artifacts/
└── melanoma_cnn.keras
```

The application will later load this file without retraining.


In [13]:

MODEL_PATH = ARTIFACT_DIR / "melanoma_cnn.keras"

if model is not None:
    model.save(MODEL_PATH)

    print("Saved model:", MODEL_PATH)
    print("File size (MB):", round(MODEL_PATH.stat().st_size / (1024**2), 2))
else:
    print("No trained model available. Serialization was not performed.")


Saved model: deployment_artifacts/melanoma_cnn.keras
File size (MB): 84.64



#  Save the Exact Preprocessing Contract

For deployment, preprocessing must be identical to the validated Week 8 pipeline.

The production contract is:

```text
OpenCV
  ↓
BGR → RGB
  ↓
Resize to 128 × 128
  ↓
float32
  ↓
pixel / 255.0
  ↓
CNN
```

There is no fitted scaler/vectorizer in this image pipeline. Therefore, the preprocessing is represented as explicit configuration metadata rather than a scikit-learn object.

If a future version introduces a fitted preprocessing object, it must be serialized separately.


In [14]:

metadata = {
    "project": "Melanoma Skin Cancer — Benign vs Malignant",
    "sprint": "Sprint 4",
    "day": "Day 1",
    "model_format": "Keras .keras",
    "model_artifact": "melanoma_cnn.keras",

    "input": {
        "height": IMG_SIZE[0],
        "width": IMG_SIZE[1],
        "channels": CHANNELS,
        "color_format": COLOR_FORMAT
    },

    "preprocessing": {
        "reader": INPUT_LIBRARY,
        "color_conversion": "BGR to RGB",
        "resize": [IMG_SIZE[0], IMG_SIZE[1]],
        "dtype": "float32",
        "normalization": NORMALIZATION
    },

    "classes": CLASS_TO_LABEL,
    "positive_class": "Malignant",

    "decision_rule": {
        "reference_threshold": DEFAULT_THRESHOLD,
        "selected_threshold": SELECTED_THRESHOLD,
        "rule": "probability_malignant >= selected_threshold -> Malignant"
    },

    "week8_reference_metrics": {
        "threshold": 0.50,
        "accuracy": 0.8755,
        "precision_malignant": 0.9243,
        "recall_malignant": 0.8180,
        "f1_malignant": 0.8679,
        "roc_auc": 0.9517,
        "pr_auc": 0.9445,
        "false_positives": 67,
        "false_negatives": 182
    },

    "week8_selected_threshold_metrics": {
        "threshold": 0.35,
        "accuracy": 0.8915,
        "precision_malignant": 0.8746,
        "recall_malignant": 0.9140,
        "f1_malignant": 0.8939,
        "roc_auc": 0.9517,
        "pr_auc": 0.9445,
        "false_positives": 131,
        "false_negatives": 86
    },

    "reproducibility": {
        "seed": SEED
    }
}

METADATA_PATH = ARTIFACT_DIR / "model_metadata.json"

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved:", METADATA_PATH)
print(json.dumps(metadata, indent=2))


Saved: deployment_artifacts/model_metadata.json
{
  "project": "Melanoma Skin Cancer \u2014 Benign vs Malignant",
  "sprint": "Sprint 4",
  "day": "Day 1",
  "model_format": "Keras .keras",
  "model_artifact": "melanoma_cnn.keras",
  "input": {
    "height": 128,
    "width": 128,
    "channels": 3,
    "color_format": "RGB"
  },
  "preprocessing": {
    "reader": "OpenCV",
    "color_conversion": "BGR to RGB",
    "resize": [
      128,
      128
    ],
    "dtype": "float32",
    "normalization": "pixel / 255.0"
  },
  "classes": {
    "Benign": 0,
    "Malignant": 1
  },
  "positive_class": "Malignant",
  "decision_rule": {
    "reference_threshold": 0.5,
    "selected_threshold": 0.35,
    "rule": "probability_malignant >= selected_threshold -> Malignant"
  },
  "week8_reference_metrics": {
    "threshold": 0.5,
    "accuracy": 0.8755,
    "precision_malignant": 0.9243,
    "recall_malignant": 0.818,
    "f1_malignant": 0.8679,
    "roc_auc": 0.9517,
    "pr_auc": 0.9445,
    "fals


# Recreate the Serving Preprocessing Function

This is the exact preprocessing logic that the future deployment application should use.

Keeping it in one explicit function reduces the chance of implementing a different preprocessing pipeline in the serving code.


In [17]:

import cv2

def preprocess_for_serving(image_path):
    image = cv2.imread(str(image_path))

    if image is None:
        raise ValueError(f"Could not read image: {image_path}")

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(
        image,
        IMG_SIZE,
        interpolation=cv2.INTER_AREA
    )
    image = image.astype(np.float32) / 255.0

    return np.expand_dims(image, axis=0)

print("Serving preprocessing function created.")


Serving preprocessing function created.



#  Save → Load → Predict Verification

This is the key serialization test.

We want to prove:

```text
Trained model
      ↓
Save
      ↓
melanoma_cnn.keras
      ↓
Load
      ↓
Prediction
```

The prediction from the reloaded model should match the prediction from the original in-memory model for the same input.


In [18]:

TEST_IMAGE_PATH = "/content/5603.jpg"
if TEST_IMAGE_PATH:
    print("Test image:", TEST_IMAGE_PATH)
else:
    print("Set TEST_IMAGE_PATH to one known Week 8 test image.")


Test image: /content/5603.jpg


In [19]:

if not MODEL_PATH.exists():
    print("Serialized model is not available yet.")
else:
    loaded_model = load_model(MODEL_PATH)
    print("Reloaded model successfully:", MODEL_PATH)

    if TEST_IMAGE_PATH and Path(TEST_IMAGE_PATH).exists():
        x = preprocess_for_serving(TEST_IMAGE_PATH)

        original_probability = float(
            np.asarray(model.predict(x, verbose=0)).squeeze()
        )

        reloaded_probability = float(
            np.asarray(loaded_model.predict(x, verbose=0)).squeeze()
        )

        original_class = (
            "Malignant"
            if original_probability >= SELECTED_THRESHOLD
            else "Benign"
        )

        reloaded_class = (
            "Malignant"
            if reloaded_probability >= SELECTED_THRESHOLD
            else "Benign"
        )

        print("Original probability :", round(original_probability, 6))
        print("Reloaded probability :", round(reloaded_probability, 6))
        print("Absolute difference  :", abs(original_probability - reloaded_probability))
        print("Original prediction  :", original_class)
        print("Reloaded prediction  :", reloaded_class)

        np.testing.assert_allclose(
            original_probability,
            reloaded_probability,
            rtol=1e-5,
            atol=1e-5
        )

        assert original_class == reloaded_class

        print("\nPASS: Save → Load → Predict reproduced the same result.")
    else:
        print("Model reload passed.")
        print("Prediction comparison will run after TEST_IMAGE_PATH is supplied.")


Reloaded model successfully: deployment_artifacts/melanoma_cnn.keras
Model reload passed.
Prediction comparison will run after TEST_IMAGE_PATH is supplied.



# Optional: Save Any Fitted Preprocessing Objects

This Week 8 image pipeline does not use a fitted `StandardScaler`, `Encoder`, or `Vectorizer`.

If such an object is introduced later, it must be saved with `joblib` and loaded by the serving application.

The rule is:
**Never load the model alone if production also depends on fitted preprocessing objects.**


In [21]:

import joblib

fitted_preprocessing_objects = {}

if fitted_preprocessing_objects:
    for name, obj in fitted_preprocessing_objects.items():
        path = ARTIFACT_DIR / f"{name}.joblib"
        joblib.dump(obj, path)
        print("Saved:", path)
else:
    print("No fitted preprocessing objects to serialize.")
    print("The image preprocessing contract is stored in model_metadata.json.")


No fitted preprocessing objects to serialize.
The image preprocessing contract is stored in model_metadata.json.



#  Reproducibility — Fixed Seeds

The Week 8 project used:

```python
SEED = 42
```

The seed is fixed for Python, NumPy, and TensorFlow.

This does not guarantee bit-for-bit identical results across every machine or GPU, but it reduces uncontrolled randomness and documents the intended experimental setup.


In [22]:

print("Python random seed:", SEED)
print("NumPy seed:", SEED)
print("TensorFlow seed:", SEED)
print("Reproducibility configuration recorded.")


Python random seed: 42
NumPy seed: 42
TensorFlow seed: 42
Reproducibility configuration recorded.



#  Pinned `requirements.txt`

Deployment should use the same library versions used during model development.

The next cell records the versions of the main packages required by this project.



In [23]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 119.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 124.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [24]:

from importlib.metadata import version, PackageNotFoundError

packages = [
    "tensorflow",
    "numpy",
    "pandas",
    "opencv-python",
    "scikit-learn",
    "joblib",
    "pillow",
    "mlflow",
]

requirements = []

for package in packages:
    try:
        requirements.append(f"{package}=={version(package)}")
    except PackageNotFoundError:
        print("Not installed:", package)

requirements_path = ARTIFACT_DIR / "requirements.txt"

with open(requirements_path, "w", encoding="utf-8") as f:
    f.write("\n".join(requirements) + "\n")

print("\nCreated:", requirements_path)
print("\n".join(requirements))



Created: deployment_artifacts/requirements.txt
tensorflow==2.20.0
numpy==2.1.3
pandas==2.2.3
opencv-python==5.0.0.93
scikit-learn==1.6.1
joblib==1.6.0
pillow==11.3.0
mlflow==3.16.0



#  MLflow — MLOps Traceability

MLflow is used here to record the important Day 1 deployment information:

- Model type
- Input size
- Preprocessing
- Threshold
- Random seed
- Week 8 evaluation metrics
- Serialized model
- Metadata
- Requirements

This makes the deployment artifact traceable instead of being just an unexplained `.keras` file.


In [25]:

try:
    import mlflow

    mlflow.set_experiment("Melanoma_Sprint4_Deployment")

    with mlflow.start_run(run_name="Day1_Serialization_MLOps"):
        mlflow.log_param("model", "CNN")
        mlflow.log_param("model_format", "Keras .keras")
        mlflow.log_param("image_size", "128x128")
        mlflow.log_param("channels", 3)
        mlflow.log_param("color_format", "RGB")
        mlflow.log_param("normalization", "pixel/255.0")
        mlflow.log_param("reference_threshold", DEFAULT_THRESHOLD)
        mlflow.log_param("selected_threshold", SELECTED_THRESHOLD)
        mlflow.log_param("seed", SEED)

        mlflow.log_metric("week8_reference_accuracy", 0.8755)
        mlflow.log_metric("week8_reference_precision_malignant", 0.9243)
        mlflow.log_metric("week8_reference_recall_malignant", 0.8180)
        mlflow.log_metric("week8_reference_f1_malignant", 0.8679)
        mlflow.log_metric("roc_auc", 0.9517)
        mlflow.log_metric("pr_auc", 0.9445)

        if MODEL_PATH.exists():
            mlflow.log_artifact(str(MODEL_PATH), artifact_path="model")

        if METADATA_PATH.exists():
            mlflow.log_artifact(str(METADATA_PATH), artifact_path="metadata")

        if requirements_path.exists():
            mlflow.log_artifact(str(requirements_path), artifact_path="environment")

    print("MLflow tracking completed successfully.")

except Exception as e:
    print("MLflow tracking was skipped or failed.")
    print(type(e).__name__, "-", e)


2026/09/13 15:55:21 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/13 15:55:21 INFO mlflow.store.db.utils: Updating database tables
2026/09/13 15:55:23 INFO mlflow.tracking.fluent: Experiment with name 'Melanoma_Sprint4_Deployment' does not exist. Creating a new experiment.


MLflow tracking completed successfully.



#  Deployment Manifest

The manifest is a compact record of the Day 1 deployment package.

It helps the next Sprint 4 day know exactly which model, preprocessing contract, threshold, and environment belong together.


In [26]:

manifest = {
    "project": "Melanoma Skin Cancer — Benign vs Malignant",
    "week": 9,
    "day": 1,
    "sprint": 4,
    "goal": "Prepare the validated Week 8 CNN for deployment",

    "artifacts": {
        "model": str(MODEL_PATH),
        "metadata": str(METADATA_PATH),
        "requirements": str(requirements_path)
    },

    "deployment_contract": {
        "image_size": [128, 128],
        "channels": 3,
        "color_format": "RGB",
        "normalization": "pixel / 255.0",
        "class_mapping": CLASS_TO_LABEL,
        "selected_threshold": SELECTED_THRESHOLD
    },

    "week8_selected_metrics": {
        "accuracy": 0.8915,
        "precision_malignant": 0.8746,
        "recall_malignant": 0.9140,
        "f1_malignant": 0.8939,
        "roc_auc": 0.9517,
        "pr_auc": 0.9445,
        "false_positives": 131,
        "false_negatives": 86
    },

    "checks": {
        "seed_fixed": True,
        "preprocessing_documented": True,
        "model_serialized": MODEL_PATH.exists(),
        "metadata_saved": METADATA_PATH.exists(),
        "requirements_saved": requirements_path.exists()
    }
}

MANIFEST_PATH = ARTIFACT_DIR / "deployment_manifest.json"

with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))


{
  "project": "Melanoma Skin Cancer \u2014 Benign vs Malignant",
  "week": 9,
  "day": 1,
  "sprint": 4,
  "goal": "Prepare the validated Week 8 CNN for deployment",
  "artifacts": {
    "model": "deployment_artifacts/melanoma_cnn.keras",
    "metadata": "deployment_artifacts/model_metadata.json",
    "requirements": "deployment_artifacts/requirements.txt"
  },
  "deployment_contract": {
    "image_size": [
      128,
      128
    ],
    "channels": 3,
    "color_format": "RGB",
    "normalization": "pixel / 255.0",
    "class_mapping": {
      "Benign": 0,
      "Malignant": 1
    },
    "selected_threshold": 0.35
  },
  "week8_selected_metrics": {
    "accuracy": 0.8915,
    "precision_malignant": 0.8746,
    "recall_malignant": 0.914,
    "f1_malignant": 0.8939,
    "roc_auc": 0.9517,
    "pr_auc": 0.9445,
    "false_positives": 131,
    "false_negatives": 86
  },
  "checks": {
    "seed_fixed": true,
    "preprocessing_documented": true,
    "model_serialized": true,
    "metadat


#  Final Day 1 Checklist

### Sprint Planning
- [x] Sprint 4 goal defined.
- [x] Deployment backlog defined.
- [x] Sprint 3 retrospective improvement carried forward.

### Serialization
- [ ] Final Week 8 CNN loaded.
- [ ] CNN saved as `melanoma_cnn.keras`.
- [ ] Saved model successfully reloaded.
- [ ] Known prediction reproduced after loading.

### Preprocessing
- [x] 128 × 128 input documented.
- [x] BGR → RGB documented.
- [x] float32 documented.
- [x] `/255.0` normalization documented.
- [x] Benign/Malignant mapping documented.
- [x] Selected threshold `0.35` documented.

### MLOps / Reproducibility
- [x] Seed = 42 documented.
- [ ] `requirements.txt` generated from the Week 8 environment.
- [ ] MLflow tracking attempted.
- [x] Deployment manifest created.

### Deployment readiness
- [x] Model artifact structure defined.
- [x] Metadata artifact defined.
- [x] Serving preprocessing function defined.
- [x] Next step identified: prediction API / application.

**Day 1 Definition of Done:** The validated Week 8 model and its exact serving contract are packaged in a reproducible form and are ready to be integrated into the Sprint 4 application.



#  Day 1 Summary

Day 1 is the bridge between **Sprint 3 model evaluation** and **Sprint 4 deployment**.
Day 1 is not about improving the CNN. It is about packaging the validated CNN correctly so that the same model and the same preprocessing can be used reliably outside the notebook.
